# Experiments & ablations

Use **`notebooks/EDA.ipynb`** for the full pipeline: environment setup, preprocessing, Model A training, Model B training, and artifact checks.

Use **this notebook** for optional extra work that belongs in a report but not in the main runner:

- Compare metrics across runs (e.g. after changing hyperparameters).
- Log tables or plots for ensemble weight sweeps, `top_k` changes, or capped training sizes.
- Paste command lines and note the Git commit hash for reproducibility.

Nothing is *missing* from the repo if this file stays minimal; it is a separate scratch space so EDA does not turn into a long experiment log.

## 1. Project root (Colab or local)

Reuse the same idea as `EDA.ipynb`: run from `race_rc_project` or let the cell locate it under `/content`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()


def find_project_root(start: Path) -> Path | None:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    for p in (
        Path("/content/race_rc_project"),
        Path("/content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project"),
    ):
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    base = Path("/content")
    if base.exists():
        for p in base.rglob("race_rc_project"):
            if (p / "requirements.txt").exists() and (p / "src").exists():
                return p
    return None


project_root = find_project_root(ROOT)
if project_root is None:
    raise FileNotFoundError("Clone the repo and open this notebook from race_rc_project (see EDA.ipynb).")
if Path.cwd().resolve() != project_root.resolve():
    %cd {project_root}
print("Project root:", Path.cwd())
print("Python:", sys.version.split()[0])

## 2. Snapshot: Model A / Model B metrics

After any training run, compare validation vs test without opening JSON by hand.

In [ ]:
import json
from pathlib import Path

def show_metrics(path: Path, title: str) -> None:
    if not path.exists():
        print(f"[missing] {title}: {path}")
        return
    data = json.loads(path.read_text(encoding="utf-8"))
    print("===", title, "===")
    print("config:", json.dumps(data.get("config", {}), indent=2))
    for split in ("validation", "test"):
        if split not in data:
            continue
        print(f"\n-- {split} --")
        block = data[split]
        if "distractor_generation" in block:
            for key in ("distractor_generation", "hint_generation"):
                print(key + ":", json.dumps(block[key], indent=2))
        else:
            print(json.dumps(block, indent=2))


show_metrics(Path("models/model_a/traditional/metrics_summary.json"), "Model A")
print()
show_metrics(Path("models/model_b/traditional/metrics_summary.json"), "Model B")

## 3. Example ablations (run in separate runs, then re-run section 2)

Ideas to document in your report (adjust flags to match your CLI — see `README.md`):

- **Ensemble balance**: train Model A or B with a different supervised weight (e.g. `0.3` vs `0.7`) and compare BLEU/ROUGE/METEOR.
- **Candidate breadth**: change how many sentences or terms are considered (if exposed in `TrainConfig` / argparse).
- **Smaller data**: cap `max_train_mcq` for a fast sanity curve vs full data.

Keep one row per run: date, commit hash, command, and path to saved `metrics_summary.json` (or copy the JSON into this notebook output).